# EDA — Indicadores e Fatores Associados a Acidentes Fatais
## Base de Acidentes da PRF — `dados_abertos_prf-datatran2025.csv`

Reprodução do roteiro de EDA do módulo de referência (base de vendas), agora aplicado à
**base de dados abertos da Polícia Rodoviária Federal (PRF)** para o ano de 2025.
Indicadores → rankings e séries → análise bivariada orientada a um alvo → combinações de
fatores → gráficos → síntese interpretativa.

**Base de dados:** `dados_abertos_prf-datatran2025.csv` — registros de acidentes de trânsito
nas rodovias federais brasileiras ocorridos em 2025, com colunas de data/hora, UF, BR, km,
município, causa e tipo do acidente, classificação de gravidade, condições da via e do
tempo, e contagens de pessoas/vítimas/veículos envolvidos.

**Problema analítico central:** *Quais fatores estão associados à ocorrência de acidentes
com vítima fatal?*

**Variável-alvo:** `acidente_fatal`
- `1` = o acidente teve **pelo menos 1 morto** (`mortos > 0`)
- `0` = o acidente não teve mortos (`mortos == 0`)

A coluna `classificacao_acidente` é usada apenas como conferência. Ela não é estritamente
equivalente ao alvo: existe um registro com morte e classificação ausente na base.

**Unidade de análise:** ocorrência de acidente registrada pela PRF (uma linha do CSV).

> Assim como no módulo de referência (onde o evento problemático era raro frente ao total de
> pedidos), aqui o evento de interesse — acidente **fatal** — também é minoritário diante do
> total de acidentes. O cuidado metodológico é o mesmo: não confundir **volume** com
> **proporção de acidentes fatais**, e sempre comparar qualquer recorte com o **percentual
> global** de referência.

In [1]:
# Importação da biblioteca PANDAS
# Essencial para leitura, organização e manipulação tabular dos dados (DataFrames)
import pandas as pd

# Importação da biblioteca NUMPY
# Usada como apoio para operações numéricas (médias, arredondamentos, tratamento de NaN)
import numpy as np

# Importação do MATPLOTLIB (módulo pyplot)
# Biblioteca de visualização usada para construir os gráficos da análise exploratória
import matplotlib.pyplot as plt

# Importação do módulo de formatação de eixos do matplotlib
# Usado para exibir os eixos dos gráficos em formato de porcentagem (%), facilitando a leitura
import matplotlib.ticker as mtick

In [2]:
# Configuração de exibição de números decimais do pandas
# Formata todos os números float exibidos em tabelas com separador de milhar e 2 casas decimais
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Ajuste da resolução (DPI) padrão dos gráficos
# Deixa os gráficos exibidos no notebook com melhor nitidez
plt.rcParams["figure.dpi"] = 110

In [3]:
# Paleta de cores usada em todos os gráficos da EDA (categórica, fixa por papel)
# Mantemos as mesmas cores ao longo do notebook para que cada categoria seja sempre
# reconhecida pela mesma cor (evita repintar séries quando um filtro muda a seleção)
C_BLUE, C_ORANGE, C_AQUA   = "#2a78d6", "#eb6834", "#1baf7a"
C_YELLOW, C_MAGENTA        = "#eda100", "#e87ba4"
C_GREEN, C_VIOLET, C_RED   = "#008300", "#4a3aa7", "#e34948"
C_INK, C_MUTED             = "#0b0b0b", "#898781"

# Dicionário de mapeamento UF -> Região
# A base da PRF não traz a coluna "região" diretamente — apenas a UF —, então
# construímos esse mapeamento manualmente para permitir rankings e séries por região
UF_REGIAO = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte",
    "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste",
    "PB": "Nordeste", "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul",
}

In [4]:
# Caminho do arquivo CSV da base de acidentes da PRF
# Leitura do arquivo CSV da base de acidentes
# A base da PRF é disponibilizada com ";" como separador de colunas e acentuação em
# Latin-1 (ISO-8859-1) — por isso os parâmetros sep=";" e encoding="latin1"
df = pd.read_csv("dados_abertos_prf-datatran2025.csv", sep=";", encoding="latin1")



# Conversão da coluna de data (formato AAAA-MM-DD) de texto para o tipo datetime
# Necessário para permitir agrupamentos por mês/ano e cálculos de série temporal
df["data_inversa"] = pd.to_datetime(df["data_inversa"])

In [5]:
# Tratamento das colunas numéricas que vêm com vírgula decimal (padrão brasileiro)
# "km", "latitude" e "longitude" chegam como texto (ex.: "225,4") — convertemos a
# vírgula para ponto e transformamos em número (float) para permitir cálculos
for col in ["km", "latitude", "longitude"]:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", ".", regex=False), errors="coerce")

# Extração da hora do acidente a partir da coluna "horario" (formato HH:MM:SS)
# Usada mais adiante para a série de acidentes por hora do dia
df["hora"] = pd.to_datetime(df["horario"], format="%H:%M:%S", errors="coerce").dt.hour

# Criação da coluna "regiao" aplicando o dicionário UF -> Região definido acima
df["regiao"] = df["uf"].map(UF_REGIAO)

# Criação da coluna "mes" no formato Ano-Mês (ex.: 2025-01)
# Agrupa cada data de acidente no seu respectivo mês, base para a série temporal
df["mes"] = df["data_inversa"].dt.to_period("M").astype(str)

In [6]:
# Exibição do formato da base (linhas, colunas)
# Primeira conferência: confirma quantos acidentes e quantas colunas foram carregados
print(df.shape)

# Exibição das 5 primeiras linhas da base
# Permite uma inspeção visual inicial da estrutura e do conteúdo dos dados
df.head(5)

(72529, 33)


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,feridos,veiculos,latitude,longitude,regional,delegacia,uop,hora,regiao,mes
0,652493,2025-01-01,quarta-feira,06:20:00,SP,116,225.00,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,...,1,2,-23.49,-46.54,SPRF-SP,DEL01-SP,UOP01-DEL01-SP,6,Sudeste,2025-01
1,652519,2025-01-01,quarta-feira,07:50:00,CE,116,546.20,PENAFORTE,Pista esburacada,Colisão frontal,...,1,6,-7.81,-39.08,SPRF-CE,DEL05-CE,UOP03-DEL05-CE,7,Nordeste,2025-01
2,652522,2025-01-01,quarta-feira,08:45:00,PR,369,88.20,CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,...,3,2,-23.18,-50.64,SPRF-PR,DEL07-PR,UOP05-DEL07-PR,8,Sul,2025-01
3,652544,2025-01-01,quarta-feira,11:00:00,PR,116,74.00,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,...,1,2,-25.37,-49.04,SPRF-PR,DEL01-PR,UOP02-DEL01-PR,11,Sul,2025-01
4,652549,2025-01-01,quarta-feira,09:30:00,MG,251,471.00,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,...,2,4,-16.47,-43.43,SPRF-MG,DEL12-MG,UOP01-DEL12-MG,9,Sudeste,2025-01


In [7]:
# Exibição dos tipos de dados (dtype) de cada coluna
# Importante para confirmar se colunas numéricas (mortos, feridos, km) e de data foram
# reconhecidas corretamente, e não interpretadas como texto (object)
print("Tipos de dados:")
print(df.dtypes)

Tipos de dados:
id                                 int64
data_inversa              datetime64[ns]
dia_semana                        object
horario                           object
uf                                object
br                                 int64
km                               float64
municipio                         object
causa_acidente                    object
tipo_acidente                     object
classificacao_acidente            object
fase_dia                          object
sentido_via                       object
condicao_metereologica            object
tipo_pista                        object
tracado_via                       object
uso_solo                          object
pessoas                            int64
mortos                             int64
feridos_leves                      int64
feridos_graves                     int64
ilesos                             int64
ignorados                          int64
feridos                            int64


In [8]:
# Contagem de valores ausentes (NaN) por coluna
# Etapa obrigatória de qualidade de dados: identifica onde existem lacunas
# antes de calcular qualquer indicador
print("Valores ausentes por coluna (apenas colunas com algum ausente):")
nulos = df.isna().sum()
print(nulos[nulos > 0])

Valores ausentes por coluna (apenas colunas com algum ausente):
classificacao_acidente     1
regional                   2
delegacia                 22
uop                       38
dtype: int64


In [9]:
# Verificação do período coberto pela base
# Mostra a menor e a maior data de acidente, delimitando a janela temporal da análise
print("Período coberto:", df["data_inversa"].min().date(), "a", df["data_inversa"].max().date())

Período coberto: 2025-01-01 a 2025-12-31


In [10]:
# Contagem de acidentes por classificação de gravidade
# Primeira visão da variável que confirma o alvo (acidente_fatal): quantos acidentes são
# Sem Vítimas, Com Vítimas Feridas ou Com Vítimas Fatais
print("Classificação do acidente (contagem):")
print(df["classificacao_acidente"].value_counts(dropna=False))

Classificação do acidente (contagem):
classificacao_acidente
Com Vítimas Feridas    56181
Sem Vítimas            11138
Com Vítimas Fatais      5209
NaN                        1
Name: count, dtype: int64


In [11]:
# Criação da variável-alvo "acidente_fatal"
# Transforma a contagem de mortos em uma variável binária (0/1) que sinaliza o evento de
# interesse — aqui, acidente com pelo menos 1 vítima fatal
df["acidente_fatal"] = (df["mortos"] > 0).astype(int)

In [12]:
# Cálculo do total de acidentes na base
# Denominador que será usado em todos os percentuais globais e segmentados
total_acidentes = len(df)

In [13]:
# Cálculo do total de acidentes com vítima fatal
# Soma da coluna binária acidente_fatal (soma de 1s = quantidade de acidentes fatais)
total_fatais = int(df["acidente_fatal"].sum())

In [14]:
# Cálculo do percentual global de acidentes fatais
# Razão entre acidentes fatais e o total de acidentes — a TAXA GLOBAL DE REFERÊNCIA
# que será comparada com todos os cruzamentos feitos nos blocos seguintes
pct_fatais = total_fatais / total_acidentes

In [19]:
# Cálculo do total de mortos e da letalidade a cada 100 acidentes
# "mortos" soma todas as vítimas fatais (um acidente pode ter mais de 1 morto);
# letalidade_100 é uma métrica de gravidade global mais fina do que só a % de acidentes fatais
total_mortos = int(df["mortos"].sum())
letalidade_100 = total_mortos / total_acidentes * 100

In [ ]:
# Cálculo de indicadores complementares de volume: feridos, ilesos, veículos e pessoas
# envolvidas — dão a dimensão humana e material do total de ocorrências
total_feridos        = int(df["feridos"].sum())
total_feridos_graves = int(df["feridos_graves"].sum())
total_ilesos          = int(df["ilesos"].sum())
total_veiculos         = int(df["veiculos"].sum())
total_pessoas          = int(df["pessoas"].sum())

In [20]:
# Montagem da tabela-resumo de indicadores globais
# Consolida todos os indicadores calculados acima em uma única tabela, pronta para
# ser incluída no relatório exploratório
indicadores_globais = pd.DataFrame({
    "indicador": [
        "Total de acidentes", "Acidentes com vítima fatal", "% de acidentes fatais",
        "Total de mortos", "Mortos por 100 acidentes",
        "Total de feridos", "Total de feridos graves", "Total de ilesos",
        "Total de veículos envolvidos", "Total de pessoas envolvidas",
        "Média de pessoas por acidente", "Média de veículos por acidente",
    ],
    "valor": [
        total_acidentes, total_fatais, f"{pct_fatais:.2%}",
        total_mortos, f"{letalidade_100:.2f}",
        total_feridos, total_feridos_graves, total_ilesos,
        total_veiculos, total_pessoas,
        f"{total_pessoas/total_acidentes:.2f}", f"{total_veiculos/total_acidentes:.2f}",
    ],
})

# Exibição da tabela final de indicadores globais
indicadores_globais

,indicador,valor
0,Total de acidentes,72529
1,Acidentes com vítima fatal,5210
2,% de acidentes fatais,7.18%
3,Total de mortos,6043
4,Mortos por 100 acidentes,8.33
5,Total de feridos,83550
6,Total de feridos graves,20018
7,Total de ilesos,76406
8,Total de veículos envolvidos,144922
9,Total de pessoas envolvidas,188346


In [21]:
# Estatística descritiva das variáveis numéricas de contagem de vítimas/veículos
# describe() traz contagem, média, desvio-padrão, mínimo, quartis e máximo;
# complementamos com mediana e assimetria (skew), úteis para variáveis de contagem
# muito concentradas em zero (ex.: "mortos")
num_cols = ["pessoas", "mortos", "feridos_leves", "feridos_graves", "ilesos", "feridos", "veiculos", "km"]
estat_descritiva = df[num_cols].describe().T
estat_descritiva["mediana"] = df[num_cols].median()
estat_descritiva["assimetria"] = df[num_cols].skew()
estat_descritiva

,count,mean,std,min,25%,50%,75%,max,mediana,assimetria
pessoas,"72,529.00",2.60,2.26,1.00,2.00,2.00,3.00,76.00,2.00,11.79
mortos,"72,529.00",0.08,0.34,0.00,0.00,0.00,0.00,16.00,0.00,7.50
feridos_leves,"72,529.00",0.88,1.04,0.00,0.00,1.00,1.00,41.00,1.00,7.25
feridos_graves,"72,529.00",0.28,0.61,0.00,0.00,0.00,0.00,22.00,0.00,5.10
ilesos,"72,529.00",1.05,1.83,0.00,0.00,1.00,1.00,71.00,1.00,14.44
feridos,"72,529.00",1.15,1.14,0.00,1.00,1.00,1.00,49.00,1.00,8.93
veiculos,"72,529.00",2.00,1.13,1.00,1.00,2.00,2.00,82.00,2.00,7.54
km,"72,529.00",260.04,227.92,0.00,76.00,192.50,411.00,"1,257.00",192.50,0.99


## BLOCO 2 — Frequências, rankings e séries simples

**Objetivo:** organizar a distribuição dos acidentes respondendo: onde, quando e em que
condições ocorrem mais acidentes — e mais acidentes fatais?

In [22]:
# Definição de uma função de ranking reutilizável
# Evita repetir código: recebe o nome de uma coluna categórica (ex.: "uf") e devolve
# a contagem de acidentes, mortos e feridos por categoria, já ordenada por volume
def ranking(var):
    t = df.groupby(var, observed=True).agg(
        acidentes=("id", "count"),   # conta quantos acidentes existem em cada grupo
        mortos=("mortos", "sum"),     # soma os mortos de cada grupo
        feridos=("feridos", "sum"),   # soma os feridos de cada grupo
    ).sort_values("acidentes", ascending=False)  # ordena do grupo com mais acidentes para o com menos
    t["pct_acidentes"] = t["acidentes"] / total_acidentes  # participação percentual de cada grupo no total
    t["pct_fatal"] = df.groupby(var, observed=True)["acidente_fatal"].mean()  # taxa de letalidade do grupo
    return t

In [23]:
# Aplicação do ranking à coluna "regiao"
# Mostra em quais regiões do país estão concentrados os acidentes e qual a letalidade de cada uma
print("Ranking por região:")
display(ranking("regiao").sort_values("acidentes", ascending=False))

Ranking por região:


,acidentes,mortos,feridos,pct_acidentes,pct_fatal
regiao,,,,,
Sudeste,23323,1477,27943,0.32,0.06
Sul,20715,1354,23490,0.29,0.06
Nordeste,16019,1939,18337,0.22,0.10
Centro-Oeste,8497,748,9228,0.12,0.07
Norte,3975,525,4552,0.05,0.11


In [24]:
# Aplicação do ranking à coluna "uf"
# Mostra em quais estados estão concentrados os acidentes — visão mais granular que região
print("Ranking por UF (todas as 27 unidades da federação):")
display(ranking("uf"))

Ranking por UF (todas as 27 unidades da federação):


,acidentes,mortos,feridos,pct_acidentes,pct_fatal
uf,,,,,
MG,9570,765,12002,0.13,0.07
SC,8186,434,9400,0.11,0.05
PR,7630,593,8537,0.11,0.07
RJ,6428,330,7659,0.09,0.05
RS,4899,327,5553,0.07,0.06
SP,4683,221,4988,0.06,0.04
BA,4108,583,5023,0.06,0.12
GO,3196,308,3557,0.04,0.08
PE,3013,336,3348,0.04,0.10


In [25]:
# Aplicação do ranking à coluna "br" (top 15 rodovias federais por volume de acidentes)
print("Top 15 rodovias (BR) por volume de acidentes:")
display(ranking("br").head(15))

Top 15 rodovias (BR) por volume de acidentes:


,acidentes,mortos,feridos,pct_acidentes,pct_fatal
br,,,,,
101,13014,760,15033,0.18,0.05
116,11021,708,12268,0.15,0.06
40,3502,214,4183,0.05,0.05
381,3496,190,4166,0.05,0.05
153,2789,282,3117,0.04,0.08
163,2519,210,2603,0.03,0.07
364,2264,173,2515,0.03,0.07
277,2157,152,2492,0.03,0.06
262,1769,169,2137,0.02,0.08


In [26]:
# Aplicação do ranking à coluna "causa_acidente" (top 15 causas por volume)
print("Top 15 causas de acidente por volume:")
display(ranking("causa_acidente").head(15))

Top 15 causas de acidente por volume:


,acidentes,mortos,feridos,pct_acidentes,pct_fatal
causa_acidente,,,,,
Ausência de reação do condutor,11469,855,12505,0.16,0.07
Reação tardia ou ineficiente do condutor,10799,597,12295,0.15,0.05
Acessar a via sem observar a presença dos outros veículos,7097,421,8736,0.10,0.06
Condutor deixou de manter distância do veículo da frente,4413,89,5001,0.06,0.02
Velocidade Incompatível,4088,470,5133,0.06,0.09
Manobra de mudança de faixa,4016,188,4873,0.06,0.04
Ingestão de álcool pelo condutor,3685,223,3219,0.05,0.05
Demais falhas mecânicas ou elétricas,3385,59,2388,0.05,0.02
Transitar na contramão,2475,961,3452,0.03,0.30


In [27]:
# Aplicação do ranking à coluna "tipo_acidente"
print("Ranking por tipo de acidente:")
display(ranking("tipo_acidente"))

Ranking por tipo de acidente:


,acidentes,mortos,feridos,pct_acidentes,pct_fatal
tipo_acidente,,,,,
Colisão traseira,14360,683,16376,0.20,0.04
Saída de leito carroçável,10209,700,11638,0.14,0.06
Colisão transversal,9306,481,12015,0.13,0.05
Colisão lateral mesmo sentido,7885,228,8839,0.11,0.03
Tombamento,6351,293,7241,0.09,0.04
Colisão com objeto,5109,323,4952,0.07,0.06
Colisão frontal,4739,1863,7596,0.07,0.29
Queda de ocupante de veículo,3450,89,4038,0.05,0.03
Atropelamento de Pedestre,3057,919,2846,0.04,0.30


In [28]:
# Ranking por dia da semana, respeitando a ordem lógica (segunda a domingo)
# em vez da ordem alfabética padrão do groupby
DIA_SEMANA_ORDEM = ["segunda-feira", "terça-feira", "quarta-feira", "quinta-feira",
                     "sexta-feira", "sábado", "domingo"]
rank_dia_semana = df.groupby("dia_semana", observed=True).agg(
    acidentes=("id", "count"), fatais=("acidente_fatal", "sum")
).reindex(DIA_SEMANA_ORDEM)
rank_dia_semana["pct_fatal"] = rank_dia_semana["fatais"] / rank_dia_semana["acidentes"]

print("Acidentes por dia da semana:")
rank_dia_semana

Acidentes por dia da semana:


,acidentes,fatais,pct_fatal
dia_semana,,,
segunda-feira,10285,655,0.06
terça-feira,9062,586,0.06
quarta-feira,9556,608,0.06
quinta-feira,9405,625,0.07
sexta-feira,11197,766,0.07
sábado,11554,953,0.08
domingo,11470,1017,0.09


In [29]:
# Criação da série mensal: agrupamento dos acidentes por mês, com quatro métricas
# calculadas simultaneamente
serie_mensal = df.groupby("mes", observed=True).agg(
    acidentes=("id", "count"),                # quantidade de acidentes no mês
    fatais=("acidente_fatal", "sum"),          # quantidade de acidentes fatais no mês
    mortos=("mortos", "sum"),                  # total de mortos no mês
    feridos=("feridos", "sum"),                # total de feridos no mês
)

# Cálculo do percentual mensal de acidentes fatais
# Permite observar se a letalidade oscila ao longo dos meses
serie_mensal["pct_fatal"] = serie_mensal["fatais"] / serie_mensal["acidentes"]

# Exibição da série mensal completa (janeiro a dezembro de 2025)
serie_mensal

,acidentes,fatais,mortos,feridos,pct_fatal
mes,,,,,
2025-01,5528,359,418,6915,0.06
2025-02,5287,362,412,5974,0.07
2025-03,5960,402,462,6784,0.07
2025-04,5786,414,495,6678,0.07
2025-05,6096,504,574,6834,0.08
2025-06,6122,457,528,6920,0.07
2025-07,6238,456,536,7129,0.07
2025-08,6246,472,554,7006,0.08
2025-09,6017,438,500,6876,0.07


In [30]:
# Criação da série por hora do dia — quando, ao longo das 24h, os acidentes e a
# letalidade se concentram
serie_hora = df.groupby("hora", observed=True).agg(
    acidentes=("id", "count"), fatais=("acidente_fatal", "sum")
)
serie_hora["pct_fatal"] = serie_hora["fatais"] / serie_hora["acidentes"]
serie_hora

,acidentes,fatais,pct_fatal
hora,,,
0,1508,166,0.11
1,1291,130,0.10
2,1160,143,0.12
3,1263,173,0.14
4,1595,207,0.13
5,2090,259,0.12
6,3169,242,0.08
7,4583,174,0.04
8,3765,155,0.04


In [31]:
# Definição da função de análise bivariada
# Função central do roteiro: recebe uma variável explicativa (ex.: "causa_acidente") e cruza
# com a variável-alvo (acidente_fatal), calculando volume, quantidade de acidentes fatais
# e o percentual de letalidade por categoria
def analise_bivariada(var, min_acidentes=30):
    tab = df.groupby(var, observed=True).agg(
        acidentes=("id", "count"),                 # total de acidentes na categoria
        fatais=("acidente_fatal", "sum"),           # total de acidentes fatais na categoria
        mortos=("mortos", "sum"),                    # total de mortos na categoria
    )
    tab["pct_fatal"] = tab["fatais"] / tab["acidentes"]  # proporção condicional de letalidade
    tab = tab[tab["acidentes"] >= min_acidentes]          # filtro de volume mínimo (evita categorias irrisórias)
    return tab.sort_values("pct_fatal", ascending=False)  # ordena da maior para a menor letalidade

# Exibição da taxa global, para servir de referência de comparação em todas as tabelas abaixo
print(f"Taxa global de referência: {pct_fatais:.1%}\n")

Taxa global de referência: 7.2%



In [32]:
# Laço de repetição que aplica a função de análise bivariada a seis variáveis explicativas
# Repete a mesma lógica de forma padronizada para causa, tipo, condição climática, fase do
# dia, traçado da via e tipo de pista
for var, min_n in [("causa_acidente", 100), ("tipo_acidente", 100),
                    ("condicao_metereologica", 30), ("fase_dia", 30),
                    ("tracado_via", 100), ("tipo_pista", 30)]:
    print(f"=== {var} x acidente_fatal (mínimo {min_n} acidentes) ===")
    display(analise_bivariada(var, min_n).head(10))
    print()

=== causa_acidente x acidente_fatal (mínimo 100 acidentes) ===


,acidentes,fatais,mortos,pct_fatal
causa_acidente,,,,
Suicídio (presumido),190,106,111,0.56
Pedestre andava na pista,606,250,252,0.41
Entrada inopinada do pedestre,667,203,209,0.30
Transitar na contramão,2475,736,961,0.30
Pedestre cruzava a pista fora da faixa,525,130,133,0.25
Pedestre - Ingestão de álcool/ substâncias psicoativas,150,27,28,0.18
Ultrapassagem Indevida,1770,302,404,0.17
Iluminação deficiente,157,23,24,0.15
Deficiência do Sistema de Iluminação/Sinalização,165,24,24,0.15



=== tipo_acidente x acidente_fatal (mínimo 100 acidentes) ===


,acidentes,fatais,mortos,pct_fatal
tipo_acidente,,,,
Atropelamento de Pedestre,3057,902,919,0.30
Colisão frontal,4739,1396,1863,0.29
Colisão lateral sentido oposto,2152,212,255,0.10
Eventos atípicos,287,23,24,0.08
Atropelamento de Animal,1133,68,74,0.06
Saída de leito carroçável,10209,605,700,0.06
Colisão com objeto,5109,297,323,0.06
Capotamento,1373,63,74,0.05
Colisão transversal,9306,427,481,0.05



=== condicao_metereologica x acidente_fatal (mínimo 30 acidentes) ===


...

,acidentes,fatais,mortos,pct_fatal
fase_dia,,,,
Amanhecer,3447,386,472,0.11
Plena Noite,24781,2522,2892,0.10
Anoitecer,3926,253,288,0.06
Pleno dia,40375,2049,2391,0.05



=== tracado_via x acidente_fatal (mínimo 100 acidentes) ===


,acidentes,fatais,mortos,pct_fatal
tracado_via,,,,
Curva;Declive,1430,165,200,0.12
Declive;Curva,1138,127,166,0.11
Aclive;Reta,1388,154,185,0.11
Ponte;Reta,136,15,17,0.11
Reta;Ponte,338,37,38,0.11
Declive;Reta,1492,151,183,0.10
Reta;Declive,2059,205,242,0.10
Curva;Aclive,604,60,73,0.10
Ponte,103,10,10,0.10



=== tipo_pista x acidente_fatal (mínimo 30 acidentes) ===


,acidentes,fatais,mortos,pct_fatal
tipo_pista,,,,
Simples,34733,3424,4143,0.10
Dupla,30782,1501,1603,0.05
Múltipla,7014,285,297,0.04


In [33]:
# Cruzamento adicional: letalidade por UF (todas com >= 30 acidentes no ano)
print("UF x acidente_fatal (ordenado pela maior letalidade):")
analise_bivariada("uf", min_acidentes=30)

UF x acidente_fatal (ordenado pela maior letalidade):


,acidentes,fatais,mortos,pct_fatal
uf,,,,
MA,1262,236,281,0.19
PA,1117,193,224,0.17
RR,142,23,28,0.16
AM,138,19,26,0.14
AL,629,86,96,0.14
TO,677,83,102,0.12
CE,1302,153,171,0.12
BA,4108,476,583,0.12
AC,280,29,29,0.10
